In [2]:
import pandas as pd
import numpy as np

# 1. Загружаем базовый датасет
df_spot = pd.read_csv("Bitcoin_history_data.csv")

# Убедимся, что столбец Date имеет правильный тип datetime для сравнения дат
df_spot['Date'] = pd.to_datetime(df_spot['Date'])

# 2. Добавляем разметку рыночных эр (market_era)
# Используем np.select, чтобы задать условия и соответствующие им названия эр
era_conditions = [
    df_spot['Date'] < '2017-12-18',  # До запуска первых фьючерсов CME
    (df_spot['Date'] >= '2017-12-18') & (df_spot['Date'] < '2024-01-11'),  # Эра фьючерсов без спотовых ETF
    df_spot['Date'] >= '2024-01-11'  # Эра спотовых Bitcoin-ETF
]
era_choices = ['Pre_Institutional', 'Early_Institutional', 'ETF_Era']
df_spot['market_era'] = np.select(era_conditions, era_choices, default='Unknown')

# 3. Добавляем разметку халвинг-циклов (halving_cycle)
# Точные даты халвингов: 09.07.2016, 11.05.2020, 19.04.2024
halving_conditions = [
    df_spot['Date'] < '2016-07-09',
    (df_spot['Date'] >= '2016-07-09') & (df_spot['Date'] < '2020-05-11'),
    (df_spot['Date'] >= '2020-05-11') & (df_spot['Date'] < '2024-04-19'),
    df_spot['Date'] >= '2024-04-19'
]
halving_choices = [1, 2, 3, 4]
df_spot['halving_cycle'] = np.select(halving_conditions, halving_choices, default=0)

# 4. Сохраняем обновленный базовый датасет под новым именем
df_spot.to_csv("Bitcoin_history_data_with_events.csv", index=False, encoding='utf-8')

# 5. Первичная проверка результата
print("✅ Событийные данные успешно добавлены в базовый датасет.")
print(f"Размер датасета: {df_spot.shape[0]} строк, {df_spot.shape[1]} столбцов")

print("\nРаспределение дней по рыночным эрам:")
print(df_spot['market_era'].value_counts())

print("\nРаспределение дней по халвинг-циклам:")
print(df_spot['halving_cycle'].value_counts().sort_index())

print("\nПервые 5 строк обновленного датасета:")
display(df_spot.head())

✅ Событийные данные успешно добавлены в базовый датасет.
Размер датасета: 4175 строк, 8 столбцов

Распределение дней по рыночным эрам:
market_era
Early_Institutional    2215
Pre_Institutional      1188
ETF_Era                 772
Name: count, dtype: int64

Распределение дней по халвинг-циклам:
halving_cycle
1     661
2    1402
3    1439
4     673
Name: count, dtype: int64

Первые 5 строк обновленного датасета:


,Date,Close,High,Low,Open,Volume,market_era,halving_cycle
0,2014-09-17,457.334015,468.174011,452.421997,465.864014,21056800,Pre_Institutional,1
1,2014-09-18,424.440002,456.859985,413.104004,456.859985,34483200,Pre_Institutional,1
2,2014-09-19,394.795990,427.834991,384.532013,424.102997,37919700,Pre_Institutional,1
3,2014-09-20,408.903992,423.295990,389.882996,394.673004,36863600,Pre_Institutional,1
4,2014-09-21,398.821014,412.425995,393.181000,408.084991,26580100,Pre_Institutional,1


In [3]:
import pandas as pd

# 1. Загружаем датасет с добавленными событийными данными
df_spot_final = pd.read_csv("Bitcoin_history_data_with_events.csv")

# 2. Обрабатываем столбец Volume: переводим в миллионы и переименовываем
if 'Volume' in df_spot_final.columns:
    # Делим на миллион и сразу округляем до 1 знака
    df_spot_final['Volume_mln'] = (df_spot_final['Volume'] / 1_000_000).round(1)
    # Удаляем исходный столбец, чтобы в таблице не было двух похожих метрик
    df_spot_final = df_spot_final.drop(columns=['Volume'])

# 3. Округляем все остальные числовые столбцы (цены) до 1 знака после запятой
price_columns = ['Open', 'High', 'Low', 'Close']
for col in price_columns:
    if col in df_spot_final.columns:
        df_spot_final[col] = df_spot_final[col].round(1)

# 4. Сохраняем финальную версию базового датасета
final_spot_file = "Bitcoin_history_data_final.csv"
df_spot_final.to_csv(final_spot_file, index=False, encoding='utf-8')

print(f"✅ Базовый датасет успешно приведен к единому формату и сохранен как: {final_spot_file}")
print("\nПервые 5 строк финального датасета (обратите внимание на Volume_mln и цены):")
display(df_spot_final.head())

print("\nИнформация о типах данных:")
df_spot_final.info()

✅ Базовый датасет успешно приведен к единому формату и сохранен как: Bitcoin_history_data_final.csv

Первые 5 строк финального датасета (обратите внимание на Volume_mln и цены):


,Date,Close,High,Low,Open,market_era,halving_cycle,Volume_mln
0,2014-09-17,457.3,468.2,452.4,465.9,Pre_Institutional,1,21.1
1,2014-09-18,424.4,456.9,413.1,456.9,Pre_Institutional,1,34.5
2,2014-09-19,394.8,427.8,384.5,424.1,Pre_Institutional,1,37.9
3,2014-09-20,408.9,423.3,389.9,394.7,Pre_Institutional,1,36.9
4,2014-09-21,398.8,412.4,393.2,408.1,Pre_Institutional,1,26.6



Информация о типах данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4175 entries, 0 to 4174
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           4175 non-null   object 
 1   Close          4175 non-null   float64
 2   High           4175 non-null   float64
 3   Low            4175 non-null   float64
 4   Open           4175 non-null   float64
 5   market_era     4175 non-null   object 
 6   halving_cycle  4175 non-null   int64  
 7   Volume_mln     4175 non-null   float64
dtypes: float64(5), int64(1), object(2)
memory usage: 261.1+ KB
